# Hướng 1 — Chọn tham số và đánh giá CF trên test

**Người thực hiện: Cường Quốc**  
Dataset: All Beauty đã làm sạch, dùng bốn cột `user_id`, `parent_asin`, `rating`, `timestamp`.

Notebook tiếp nối `02_item_based_cf.ipynb`. Kết quả gồm cấu hình được chọn trên validation, bảng đánh giá test của Most Popular và ItemCF, mô hình đã lưu và bản tổng hợp để viết báo cáo.

## Quy ước đánh giá được chốt trước

- Tương tác tích cực: rating từ 4 sao trở lên theo `split_config.json`. Đây là quy ước từ dữ liệu review.
- Học trên train, thử số sản phẩm lân cận **N = 10, 20, 50, 100** trên validation. Số sản phẩm trả về là **K = 10 hoặc 20**; N và K là hai tham số khác nhau.
- Chọn N có **NDCG@10 cao nhất trên toàn bộ người dùng validation đủ điều kiện đánh giá**. Nếu bằng nhau, chọn Recall@10 cao hơn, rồi N nhỏ hơn. Lưu quyết định trước khi đọc test.
- Huấn luyện lại bằng **train + validation**, giữ nguyên N đã chọn; đánh giá test và báo cáo cả hai mô hình.
- Mỗi người trong tập đánh giá phải có ít nhất một sản phẩm tích cực chưa đánh giá trong lịch sử. Tính trung bình theo người dùng, kể cả người không có hit. Báo cáo thêm nhóm đã có ít nhất một tương tác tích cực trong lịch sử.
- Hai mô hình dùng cùng tập ứng viên: toàn bộ sản phẩm xuất hiện trong lịch sử huấn luyện. Loại sản phẩm người dùng từng đánh giá ở bất kỳ mức sao nào. Sản phẩm mới trong tập đánh giá vẫn ở ground truth và được tính là miss nếu mô hình không gợi ý được.

Giữ nguyên tập test cho đánh giá cuối. Nếu xây dựng phương pháp mới sau khi xem kết quả này, ghi rõ đó là một thí nghiệm mới; không chọn lại tham số theo kết quả test hiện tại.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import scipy
import sklearn

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data' / 'processed'
REPORT_DIR = PROJECT_DIR / 'reports' / 'final_cf'
MODEL_DIR = PROJECT_DIR / 'models' / 'item_cf_final'
SELECTION_PATH = REPORT_DIR / 'selected_config.json'

CORE_COLUMNS = ['user_id', 'parent_asin', 'rating', 'timestamp']
NEIGHBOR_GRID = (10, 20, 50, 100)
BLOCK_SIZE = 1024
K_VALUES = (10, 20)
PRIMARY_K = 10
BOOTSTRAP_RESAMPLES = 1000
SEED = 42
CF_LABEL = 'ItemCF + Popular fallback'
POPULAR_LABEL = 'Most Popular'


def write_json(path, value):
    def convert(obj):
        if isinstance(obj, np.generic):
            return obj.item()
        raise TypeError(f'Không thể ghi kiểu {type(obj).__name__} ra JSON.')
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                    allow_nan=False, default=convert), encoding='utf-8')


print('Thư mục dự án:', PROJECT_DIR)
print('Python:', platform.python_version())
print('pandas:', pd.__version__, '| scipy:', scipy.__version__, '| sklearn:', sklearn.__version__)
print('N thử trên validation:', NEIGHBOR_GRID, '| K trả về:', K_VALUES)


Thư mục dự án: d:\Tài Liệu\RecommenderSystem
Python: 3.12.10
pandas: 3.0.6 | scipy: 1.18.1 | sklearn: 1.9.1
N thử trên validation: (10, 20, 50, 100) | K trả về: (10, 20)


## 1. Đọc train và validation

Sử dụng đúng các mốc thời gian đã lưu ở notebook 01. Bước này chỉ đọc train và validation; test được đọc sau khi đã lưu cấu hình được chọn.


In [2]:
required = [DATA_DIR / 'train.parquet', DATA_DIR / 'validation.parquet',
            DATA_DIR / 'split_config.json']
missing = [p.name for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Thiếu ' + ', '.join(missing)
                            + '. Đặt notebook trong thư mục dự án và chạy bước chia tập ở notebook 01.')

split_config = json.loads((DATA_DIR / 'split_config.json').read_text(encoding='utf-8'))
if split_config.get('split_method') != 'global_chronological':
    raise ValueError('Notebook này sử dụng cách chia global_chronological của notebook 01.')
if split_config.get('item_column') != 'parent_asin':
    raise ValueError('Cột sản phẩm cần là parent_asin để khớp notebook 01 và 02.')
POSITIVE_THRESHOLD = float(split_config['positive_rating_threshold'])
if POSITIVE_THRESHOLD != 4.0:
    raise ValueError('Thí nghiệm này chốt rating >= 4; cần khớp cấu hình notebook 01 và 02.')


def validate_frame(frame, name):
    if frame.empty or frame[CORE_COLUMNS].isna().any().any():
        raise ValueError(f'{name}: tập rỗng hoặc thiếu giá trị ở cột lõi.')
    if not frame['rating'].isin([1, 2, 3, 4, 5]).all():
        raise ValueError(f'{name}: rating cần thuộc 1, 2, 3, 4, 5.')
    if not pd.api.types.is_integer_dtype(frame['timestamp']):
        raise ValueError(f'{name}: timestamp cần là số nguyên mili giây.')
    if not frame['timestamp'].between(10**11, 10**14).all():
        raise ValueError(f'{name}: kiểm tra lại đơn vị timestamp mili giây.')
    if frame.duplicated(['user_id', 'parent_asin', 'timestamp']).any():
        raise ValueError(f'{name}: có sự kiện trùng user, item và timestamp.')


train = pd.read_parquet(DATA_DIR / 'train.parquet', columns=CORE_COLUMNS)
val = pd.read_parquet(DATA_DIR / 'validation.parquet', columns=CORE_COLUMNS)
for name, frame in [('train', train), ('validation', val)]:
    validate_frame(frame, name)
assert train['timestamp'].max() <= split_config['train_end_timestamp_ms']
assert val['timestamp'].min() > split_config['train_end_timestamp_ms']
assert val['timestamp'].max() <= split_config['validation_end_timestamp_ms']
assert train['timestamp'].max() < val['timestamp'].min()

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame([
    {'Tập': name, 'Dòng': len(frame), 'Người dùng': frame.user_id.nunique(),
     'Sản phẩm': frame.parent_asin.nunique()}
    for name, frame in [('Train', train), ('Validation', val)]
]))


,Tập,Dòng,Người dùng,Sản phẩm
0,Train,485977,448900,78488
1,Validation,138850,129945,39637


## 2. Khai báo mô hình và hàm đánh giá

ItemCF sử dụng ma trận nhị phân của các cặp user–item tích cực khác nhau. Độ tương tự sản phẩm là cosine giữa các vector người dùng. Mỗi sản phẩm giữ tối đa N sản phẩm lân cận có cosine dương; điểm ứng viên là tổng độ tương tự với các sản phẩm người dùng đã đánh giá tích cực. Khi thiếu ứng viên CF, danh sách được bổ sung bằng Most Popular từ cùng lịch sử.

Tính cosine theo khối ma trận thưa để phù hợp máy RAM 16 GB với dữ liệu này. Không tạo ma trận đặc có kích thước số sản phẩm nhân số sản phẩm. Thứ tự khi bằng điểm được cố định theo `parent_asin`.

`CF_slot_fraction` là tỷ lệ vị trí do CF cung cấp. `popular_only_list_fraction` là tỷ lệ người dùng nhận danh sách hoàn toàn từ Most Popular. Các chỉ số được lưu ở thang 0–1; CatalogCoverage là tỷ lệ sản phẩm xuất hiện trong ít nhất một danh sách trên danh mục của giai đoạn đó.

Ô mã dưới đây chỉ khai báo lớp và hàm; các ô tiếp theo mới chạy huấn luyện và đánh giá.


In [3]:
from pathlib import Path
from collections import defaultdict
from time import perf_counter
import hashlib
import json
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import normalize


class ItemCFModel:
    """Binary item cosine CF with train-only popularity fallback."""

    def __init__(self, n_neighbors=50, positive_threshold=4.0, block_size=1024):
        if n_neighbors < 1 or block_size < 1:
            raise ValueError('Số lân cận và kích thước khối phải dương.')
        self.n_neighbors = int(n_neighbors)
        self.positive_threshold = float(positive_threshold)
        self.block_size = int(block_size)

    def fit(self, history):
        started = perf_counter()
        columns = ['user_id', 'parent_asin', 'rating', 'timestamp']
        if history.empty or history[columns].isna().any().any():
            raise ValueError('Lịch sử huấn luyện rỗng hoặc thiếu cột lõi.')
        if not history['rating'].isin([1, 2, 3, 4, 5]).all():
            raise ValueError('Lịch sử có rating không hợp lệ.')
        positive = history[history['rating'] >= self.positive_threshold].drop_duplicates(
            ['user_id', 'parent_asin'])
        if positive.empty:
            raise ValueError('Lịch sử không có tương tác tích cực để học CF.')
        self.item_ids = pd.Index(sorted(history['parent_asin'].unique()), name='parent_asin')
        self.user_ids = pd.Index(positive['user_id'].unique(), name='user_id')
        self.user_to_index = {u: i for i, u in enumerate(self.user_ids)}
        self.warm_users = set(self.user_ids)
        self.catalog = set(self.item_ids)
        self.seen = defaultdict(set)
        for user, item in history[['user_id', 'parent_asin']].itertuples(index=False, name=None):
            self.seen[user].add(item)
        self.popularity = (positive.groupby('parent_asin').size()
            .reindex(self.item_ids, fill_value=0).rename('positive_user_count').reset_index()
            .sort_values(['positive_user_count', 'parent_asin'], ascending=[False, True])
            .reset_index(drop=True))
        self.popular_ids = self.popularity['parent_asin'].tolist()
        rows = self.user_ids.get_indexer(positive['user_id'])
        cols = self.item_ids.get_indexer(positive['parent_asin'])
        self.R = sparse.csr_matrix(
            (np.ones(len(positive), dtype=np.float32), (rows, cols)),
            shape=(len(self.user_ids), len(self.item_ids)), dtype=np.float32)
        assert self.R.nnz == len(positive)
        assert np.all(self.R.data == 1)
        vectors = normalize(self.R.T.tocsr(), norm='l2', axis=1, copy=True)
        right = vectors.T.tocsc()
        index_chunks, value_chunks, indptr = [], [], [0]
        for start in range(0, len(self.item_ids), self.block_size):
            stop = min(start + self.block_size, len(self.item_ids))
            block = (vectors[start:stop] @ right).tocsr()
            block.sum_duplicates()
            for local in range(stop - start):
                a, b = block.indptr[local:local + 2]
                indices, values = block.indices[a:b], block.data[a:b]
                keep = (indices != start + local) & (values > 0)
                indices, values = indices[keep], values[keep]
                order = np.lexsort((indices, -values))[:self.n_neighbors]
                if len(order):
                    index_chunks.append(indices[order].astype(np.int32, copy=False))
                    value_chunks.append(values[order].astype(np.float32, copy=False))
                indptr.append(indptr[-1] + len(order))
        indices = np.concatenate(index_chunks) if index_chunks else np.array([], dtype=np.int32)
        values = np.concatenate(value_chunks) if value_chunks else np.array([], dtype=np.float32)
        self.S = sparse.csr_matrix(
            (values, indices, np.asarray(indptr, dtype=np.int64)),
            shape=(len(self.item_ids), len(self.item_ids)), dtype=np.float32)
        self.S.sort_indices()
        self.S.eliminate_zeros()
        assert np.all(self.S.diagonal() == 0)
        assert np.diff(self.S.indptr).max(initial=0) <= self.n_neighbors
        assert np.isfinite(self.S.data).all()
        assert np.all((self.S.data > 0) & (self.S.data <= 1.000001))
        self.training_rows = int(len(history))
        self.positive_pairs = int(len(positive))
        self.history_end_timestamp_ms = int(history['timestamp'].max())
        self.training_fingerprint = hashlib.sha256(
            pd.util.hash_pandas_object(history[columns], index=False).values.tobytes()
        ).hexdigest()
        self.fit_seconds = perf_counter() - started
        return self

    def recommend_popular(self, user_id, k=10, return_info=False):
        if k < 1:
            raise ValueError('k phải lớn hơn hoặc bằng 1.')
        seen, result = self.seen.get(user_id, set()), []
        for item in self.popular_ids:
            if item not in seen:
                result.append(item)
                if len(result) == k:
                    break
        return (result, 0) if return_info else result

    def recommend(self, user_id, k=10, return_info=False):
        if k < 1:
            raise ValueError('k phải lớn hơn hoặc bằng 1.')
        seen, result = self.seen.get(user_id, set()), []
        index = self.user_to_index.get(user_id)
        if index is not None:
            scores = (self.R[index:index + 1] @ self.S).tocsr()
            candidates, values = scores.indices, scores.data
            keep = np.array([value > 0 and self.item_ids[i] not in seen
                             for i, value in zip(candidates, values)], dtype=bool)
            candidates, values = candidates[keep], values[keep]
            order = np.lexsort((candidates, -values))[:k]
            result = [self.item_ids[i] for i in candidates[order]]
        cf_count = len(result)
        if len(result) < k:
            chosen = set(result)
            for item in self.popular_ids:
                if item not in seen and item not in chosen:
                    result.append(item)
                    chosen.add(item)
                    if len(result) == k:
                        break
        return (result, cf_count) if return_info else result

    def save(self, destination, extra_config=None):
        destination = Path(destination)
        destination.mkdir(parents=True, exist_ok=True)
        sparse.save_npz(destination / 'user_item_train.npz', self.R)
        sparse.save_npz(destination / 'item_neighbors.npz', self.S)
        pd.DataFrame({'item_index': np.arange(len(self.item_ids)),
                      'parent_asin': self.item_ids}).to_csv(destination / 'items.csv', index=False)
        pd.DataFrame({'user_index': np.arange(len(self.user_ids)),
                      'user_id': self.user_ids}).to_csv(
            destination / 'users.csv.gz', index=False, compression='gzip')
        self.popularity.to_csv(destination / 'popularity.csv', index=False)
        seen_rows = ((user, item) for user, items in self.seen.items() for item in sorted(items))
        pd.DataFrame(seen_rows, columns=['user_id', 'parent_asin']).to_csv(
            destination / 'seen_history.csv.gz', index=False, compression='gzip')
        config = {'n_neighbors': self.n_neighbors, 'positive_threshold': self.positive_threshold,
                  'block_size': self.block_size, 'training_rows': self.training_rows,
                  'positive_pairs': self.positive_pairs, 'history_end_timestamp_ms': self.history_end_timestamp_ms,
                  'training_fingerprint': self.training_fingerprint, 'fit_seconds': self.fit_seconds,
                  'extra': extra_config or {}}
        (destination / 'model_config.json').write_text(
            json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')

    @classmethod
    def load(cls, directory):
        directory = Path(directory)
        cfg = json.loads((directory / 'model_config.json').read_text(encoding='utf-8'))
        obj = cls(cfg['n_neighbors'], cfg['positive_threshold'], cfg['block_size'])
        obj.R = sparse.load_npz(directory / 'user_item_train.npz').tocsr()
        obj.S = sparse.load_npz(directory / 'item_neighbors.npz').tocsr()
        items = pd.read_csv(directory / 'items.csv', dtype={'parent_asin': 'string'},
                            keep_default_na=False).sort_values('item_index')
        users = pd.read_csv(directory / 'users.csv.gz', dtype={'user_id': 'string'},
                            keep_default_na=False).sort_values('user_index')
        obj.item_ids = pd.Index(items['parent_asin'], name='parent_asin')
        obj.user_ids = pd.Index(users['user_id'], name='user_id')
        obj.user_to_index = {u: i for i, u in enumerate(obj.user_ids)}
        obj.warm_users, obj.catalog = set(obj.user_ids), set(obj.item_ids)
        obj.popularity = pd.read_csv(directory / 'popularity.csv', dtype={'parent_asin': 'string'},
                                     keep_default_na=False)
        obj.popular_ids = obj.popularity['parent_asin'].tolist()
        obj.seen = defaultdict(set)
        seen = pd.read_csv(directory / 'seen_history.csv.gz',
                           dtype={'user_id': 'string', 'parent_asin': 'string'}, keep_default_na=False)
        for user, item in seen.itertuples(index=False, name=None):
            obj.seen[user].add(item)
        for key in ['training_rows','positive_pairs','history_end_timestamp_ms','training_fingerprint','fit_seconds']:
            setattr(obj, key, cfg[key])
        assert obj.R.shape == (len(obj.user_ids), len(obj.item_ids))
        assert obj.S.shape == (len(obj.item_ids), len(obj.item_ids))
        return obj


def make_truth(held_out, model):
    positive = held_out[held_out['rating'] >= model.positive_threshold][
        ['user_id', 'parent_asin']].drop_duplicates()
    truth, excluded = defaultdict(set), 0
    for user, item in positive.itertuples(index=False, name=None):
        if item in model.seen.get(user, set()):
            excluded += 1
        else:
            truth[user].add(item)
    if not truth:
        raise ValueError('Tập đánh giá không còn cặp liên quan sau khi loại sản phẩm đã đánh giá.')
    diagnostics = {'users':len(truth), 'users_with_positive_history':sum(u in model.warm_users for u in truth),
                   'novel_positive_pairs':sum(map(len,truth.values())),
                   'excluded_seen_pairs':excluded,
                   'positive_pairs_with_items_unseen_in_history':sum(
                       item not in model.catalog for items in truth.values() for item in items)}
    return truth, diagnostics


def evaluate_model(model, truth, label, use_cf=True, k_values=(10,20), progress=False):
    started = perf_counter()
    recommender = model.recommend if use_cf else model.recommend_popular
    discounts = 1 / np.log2(np.arange(2,max(k_values)+2))
    rows, coverage = [], defaultdict(set)
    for position,(user,relevant) in enumerate(truth.items(),start=1):
        recs,cf_count = recommender(user,k=max(k_values),return_info=True)
        assert len(recs)<=max(k_values) and len(recs)==len(set(recs))
        assert set(recs).isdisjoint(model.seen.get(user,set()))
        assert set(recs).issubset(model.catalog)
        warm = user in model.warm_users
        for k in k_values:
            chosen=recs[:k]
            hits=np.array([item in relevant for item in chosen],dtype=float)
            hit_count=hits.sum()
            ideal=discounts[:min(k,len(relevant))].sum()
            rows.append({'model':label,'user_id':user,'k':k,'has_positive_history':warm,
                         'precision':hit_count/k,'recall':hit_count/len(relevant),
                         'ndcg':float((hits*discounts[:len(hits)]).sum()/ideal),
                         'cf_count':min(cf_count,k)})
            coverage[(k,'Toàn bộ')].update(chosen)
            if warm: coverage[(k,'Có lịch sử tích cực')].update(chosen)
        if progress and position%25000==0:
            print(f'{label}: {position:,}/{len(truth):,} người dùng')
    per_user=pd.DataFrame(rows)
    summaries=[]
    for k in k_values:
        selected=per_user[per_user.k==k]
        for cohort,group in [('Toàn bộ',selected),('Có lịch sử tích cực',selected[selected.has_positive_history])]:
            summaries.append({'model':label,'k':k,'group':cohort,'users':len(group),
                              'Precision@K':group.precision.mean(),'Recall@K':group.recall.mean(),
                              'NDCG@K':group.ndcg.mean(),
                              'CatalogCoverage@K':len(coverage[(k,cohort)])/len(model.item_ids),
                              'CF_slot_fraction':(group.cf_count/k).mean(),
                              'popular_only_list_fraction':group.cf_count.eq(0).mean()})
    return pd.DataFrame(summaries),per_user,perf_counter()-started


def paired_bootstrap_delta(cf_rows,popular_rows,k=10,n_resamples=1000,seed=42):
    left=cf_rows[cf_rows.k==k][['user_id','has_positive_history','ndcg']]
    right=popular_rows[popular_rows.k==k][['user_id','ndcg']]
    paired=left.merge(right,on='user_id',how='inner',suffixes=('_cf','_popular'),validate='one_to_one')
    assert len(paired)==len(left)==len(right)
    result=[]
    for cohort,group in [('Toàn bộ',paired),('Có lịch sử tích cực',paired[paired.has_positive_history])]:
        delta=(group.ndcg_cf-group.ndcg_popular).to_numpy()
        if not len(delta):continue
        rng=np.random.default_rng(seed)
        means=np.empty(n_resamples,dtype=float)
        for b in range(n_resamples):
            indices=rng.integers(0,len(delta),size=len(delta))
            means[b]=delta[indices].mean()
        lo,hi=np.quantile(means,[.025,.975])
        result.append({'group':cohort,'users':len(delta),'k':k,
                       'mean_NDCG_delta_CF_minus_Popular':float(delta.mean()),
                       'CI95_low':float(lo),'CI95_high':float(hi),'bootstrap_resamples':n_resamples,
                       'CI_contains_zero':bool(lo<=0<=hi)})
    return pd.DataFrame(result)


## 3. Thử các giá trị N trên validation

Mỗi cấu hình được học hoàn toàn từ train. Bảng `validation_grid.csv` gồm cả Most Popular để có mốc so sánh. Chỉ dùng NDCG@10 của nhóm **Toàn bộ** để chọn cấu hình; các cột và nhóm khác dùng để phân tích.


In [4]:
validation_summaries, tuning_runtime = [], []
first_model = ItemCFModel(NEIGHBOR_GRID[0], POSITIVE_THRESHOLD, BLOCK_SIZE).fit(train)
validation_truth, validation_diagnostics = make_truth(val, first_model)
validation_diagnostics['history_catalog_items'] = len(first_model.item_ids)
validation_diagnostics['history_rows'] = len(train)
validation_diagnostics['positive_threshold'] = POSITIVE_THRESHOLD

popular_validation, popular_validation_rows, seconds = evaluate_model(
    first_model, validation_truth, POPULAR_LABEL, use_cf=False, k_values=K_VALUES)
popular_validation['n_neighbors'] = 0  # Không áp dụng tham số N cho Most Popular.
validation_summaries.append(popular_validation)
del popular_validation_rows

for n in NEIGHBOR_GRID:
    print(f'Đang kiểm tra N={n} trên validation...', flush=True)
    trial_model = first_model if n == NEIGHBOR_GRID[0] else ItemCFModel(
        n, POSITIVE_THRESHOLD, BLOCK_SIZE).fit(train)
    summary, user_rows, evaluation_seconds = evaluate_model(
        trial_model, validation_truth, CF_LABEL, use_cf=True, k_values=K_VALUES)
    summary['n_neighbors'] = n
    validation_summaries.append(summary)
    tuning_runtime.append({
        'n_neighbors': n, 'fit_seconds': trial_model.fit_seconds,
        'evaluation_seconds': evaluation_seconds, 'neighbor_edges': trial_model.S.nnz,
        'items_with_neighbors': int((np.diff(trial_model.S.indptr) > 0).sum())})
    if n == NEIGHBOR_GRID[0]:
        train_fingerprint = trial_model.training_fingerprint
        del first_model
    del trial_model, user_rows
    gc.collect()

validation_results = pd.concat(validation_summaries, ignore_index=True)
validation_results.to_csv(REPORT_DIR / 'validation_grid.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(tuning_runtime).to_csv(REPORT_DIR / 'tuning_runtime.csv', index=False)
write_json(REPORT_DIR / 'validation_diagnostics.json', validation_diagnostics)

display(validation_results[
    (validation_results.k == PRIMARY_K) & (validation_results['group'] == 'Toàn bộ')
][['model', 'n_neighbors', 'users', 'Precision@K', 'Recall@K', 'NDCG@K',
   'CatalogCoverage@K', 'CF_slot_fraction', 'popular_only_list_fraction']].round(8))
print('Chẩn đoán validation:', validation_diagnostics)


Đang kiểm tra N=10 trên validation...
Đang kiểm tra N=20 trên validation...
Đang kiểm tra N=50 trên validation...
Đang kiểm tra N=100 trên validation...


,model,n_neighbors,users,Precision@K,Recall@K,NDCG@K,CatalogCoverage@K,CF_slot_fraction,popular_only_list_fraction
0,Most Popular,0,85951,0.001437,0.014173,0.006654,0.000140,0.000000,1.000000
4,ItemCF + Popular fallback,10,85951,0.001437,0.014132,0.006658,0.089810,0.022088,0.965189
8,ItemCF + Popular fallback,20,85951,0.001437,0.014132,0.006657,0.090103,0.022196,0.965189
12,ItemCF + Popular fallback,50,85951,0.001437,0.014132,0.006657,0.090116,0.022196,0.965189
16,ItemCF + Popular fallback,100,85951,0.001437,0.014132,0.006656,0.090141,0.022196,0.965189


Chẩn đoán validation: {'users': 85951, 'users_with_positive_history': 4392, 'novel_positive_pairs': 91672, 'excluded_seen_pairs': 18, 'positive_pairs_with_items_unseen_in_history': 41801, 'history_catalog_items': 78488, 'history_rows': 485977, 'positive_threshold': 4.0}


## 4. Chốt và lưu cấu hình trước khi đọc test

`selected_config.json` ghi lưới tham số, quy tắc chọn, mốc chia dữ liệu, cấu hình được chọn và điểm validation. Most Popular vẫn được giữ trong đánh giá cuối, kể cả khi chênh lệch validation rất nhỏ.


In [5]:
ranked = validation_results[
    (validation_results.model == CF_LABEL)
    & (validation_results.k == PRIMARY_K)
    & (validation_results['group'] == 'Toàn bộ')
].sort_values(['NDCG@K', 'Recall@K', 'n_neighbors'], ascending=[False, False, True])
if ranked.empty or not np.isfinite(ranked[['NDCG@K', 'Recall@K']].to_numpy()).all():
    raise ValueError('Không có kết quả validation hợp lệ để chọn cấu hình.')
selected_row = ranked.iloc[0]
selected_n = int(selected_row['n_neighbors'])
selection = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'algorithm': 'binary_item_cosine_with_popular_fallback',
    'neighbor_grid': list(NEIGHBOR_GRID),
    'selected_n_neighbors': selected_n,
    'positive_rating_threshold': POSITIVE_THRESHOLD,
    'block_size': BLOCK_SIZE, 'k_values': list(K_VALUES),
    'primary_metric': 'NDCG@10', 'primary_group': 'Toàn bộ',
    'tie_breakers': ['Recall@10 descending', 'n_neighbors ascending'],
    'selection_data': 'validation', 'final_fit_data': 'train + validation',
    'selected_validation_metrics': selected_row.to_dict(),
    'split_config': split_config, 'train_fingerprint': train_fingerprint,
    'validation_fingerprint': hashlib.sha256(
        pd.util.hash_pandas_object(val[CORE_COLUMNS], index=False).values.tobytes()).hexdigest(),
    'versions': {'python': platform.python_version(), 'pandas': pd.__version__,
                 'numpy': np.__version__, 'scipy': scipy.__version__,
                 'sklearn': sklearn.__version__}
}
write_json(SELECTION_PATH, selection)
selection_sha256 = hashlib.sha256(SELECTION_PATH.read_bytes()).hexdigest()
print(f'Đã chốt N={selected_n} bằng validation; đã lưu {SELECTION_PATH.name}.')
print(f"NDCG@10 validation: {selected_row['NDCG@K']:.8f}")
print('Tiếp theo: huấn luyện lại bằng train + validation với N đã chốt.')


Đã chốt N=10 bằng validation; đã lưu selected_config.json.
NDCG@10 validation: 0.00665796
Tiếp theo: huấn luyện lại bằng train + validation với N đã chốt.


## 5. Huấn luyện lại bằng train + validation

Lịch sử này kết thúc ở mốc cuối validation. Cập nhật cả cosine, danh mục sản phẩm, độ phổ biến và tập sản phẩm đã đánh giá theo lịch sử mới. Do danh mục và người dùng thay đổi, độ phủ và điểm test cần được so sánh giữa hai mô hình trên cùng test; không lấy chênh lệch giữa validation và test làm mức cải thiện.


In [6]:
assert hashlib.sha256(SELECTION_PATH.read_bytes()).hexdigest() == selection_sha256
selection = json.loads(SELECTION_PATH.read_text(encoding='utf-8'))
history = pd.concat([train, val], ignore_index=True).sort_values('timestamp', kind='stable')
history = history.reset_index(drop=True)
validate_frame(history, 'train + validation')
assert history.timestamp.max() <= split_config['validation_end_timestamp_ms']

final_model = ItemCFModel(
    selection['selected_n_neighbors'], selection['positive_rating_threshold'],
    selection['block_size']).fit(history)
final_training_info = {
    'rows': len(history), 'users_with_any_history': history.user_id.nunique(),
    'users_with_positive_history': len(final_model.user_ids),
    'catalog_items': len(final_model.item_ids), 'positive_pairs': final_model.positive_pairs,
    'neighbor_edges': final_model.S.nnz, 'fit_seconds': final_model.fit_seconds,
    'history_end_timestamp_ms': final_model.history_end_timestamp_ms,
    'training_fingerprint': final_model.training_fingerprint
}
write_json(REPORT_DIR / 'final_training_info.json', final_training_info)
print('Đã huấn luyện mô hình cuối:', final_training_info)
del history, train, val, validation_truth, validation_summaries
gc.collect()


Đã huấn luyện mô hình cuối: {'rows': 624827, 'users_with_any_history': 571069, 'users_with_positive_history': 414462, 'catalog_items': 102963, 'positive_pairs': 448557, 'neighbor_edges': 81689, 'fit_seconds': 8.710459800000535, 'history_end_timestamp_ms': 1643201980125, 'training_fingerprint': 'a99e00b28ecfc4d2254b6b653015e82c576e8d91350267ce0e2f7c2a7e0ac903'}


32

## 6. Đọc test và đánh giá hai mô hình

Tập test chỉ dùng cho đánh giá sau khi chốt N. Với mỗi người dùng, lịch sử được cố định ở cuối validation; không cập nhật bằng các tương tác phát sinh trong test. Đây là đánh giá theo lịch sử cố định, không phải mô phỏng cập nhật trực tuyến sau từng sự kiện.


In [7]:
assert hashlib.sha256(SELECTION_PATH.read_bytes()).hexdigest() == selection_sha256
test_path = DATA_DIR / 'test.parquet'
if not test_path.is_file():
    raise FileNotFoundError('Thiếu test.parquet. Chạy ô chia tập ở notebook 01.')
test = pd.read_parquet(test_path, columns=CORE_COLUMNS)
validate_frame(test, 'test')
assert test.timestamp.min() > split_config['validation_end_timestamp_ms']
assert final_model.history_end_timestamp_ms < test.timestamp.min()

test_truth, test_diagnostics = make_truth(test, final_model)
test_diagnostics.update({
    'test_rows': len(test), 'history_catalog_items': len(final_model.item_ids),
    'history_end_timestamp_ms': final_model.history_end_timestamp_ms,
    'test_start_timestamp_ms': int(test.timestamp.min()),
    'test_end_timestamp_ms': int(test.timestamp.max()),
    'selection_sha256': selection_sha256,
    'test_fingerprint': hashlib.sha256(
        pd.util.hash_pandas_object(test[CORE_COLUMNS], index=False).values.tobytes()).hexdigest()
})
write_json(REPORT_DIR / 'test_diagnostics.json', test_diagnostics)
print('Chẩn đoán test:', test_diagnostics)


Chẩn đoán test: {'users': 43656, 'users_with_positive_history': 2533, 'novel_positive_pairs': 46222, 'excluded_seen_pairs': 20, 'positive_pairs_with_items_unseen_in_history': 16505, 'test_rows': 69425, 'history_catalog_items': 102963, 'history_end_timestamp_ms': 1643201980125, 'test_start_timestamp_ms': 1643202300263, 'test_end_timestamp_ms': 1694219976666, 'selection_sha256': '6baab18c01c0f44d1e86bec12847494e10662d297414541d8b12a41bbd4df431', 'test_fingerprint': 'cdf737d38647a5d8c00f96f17a6d012b3443c6448e731d436a42f939cc99cc24'}


In [8]:
print('Đánh giá Most Popular...', flush=True)
test_popular_summary, test_popular_rows, popular_seconds = evaluate_model(
    final_model, test_truth, POPULAR_LABEL, use_cf=False, k_values=K_VALUES, progress=True)
print('Đánh giá ItemCF + Popular fallback...', flush=True)
test_cf_summary, test_cf_rows, cf_seconds = evaluate_model(
    final_model, test_truth, CF_LABEL, use_cf=True, k_values=K_VALUES, progress=True)

test_popular_summary['n_neighbors'] = 0
test_cf_summary['n_neighbors'] = final_model.n_neighbors
test_results = pd.concat([test_popular_summary, test_cf_summary], ignore_index=True)
test_per_user = pd.concat([test_popular_rows, test_cf_rows], ignore_index=True)
test_results.to_csv(REPORT_DIR / 'test_comparison.csv', index=False, encoding='utf-8-sig')
test_per_user.to_csv(REPORT_DIR / 'test_per_user.csv.gz', index=False, compression='gzip')
write_json(REPORT_DIR / 'test_runtime.json',
           {'popular_evaluation_seconds': popular_seconds, 'cf_evaluation_seconds': cf_seconds})
display(test_results[['model', 'k', 'group', 'users', 'Precision@K', 'Recall@K',
                      'NDCG@K', 'CatalogCoverage@K', 'CF_slot_fraction',
                      'popular_only_list_fraction']].round(8))


Đánh giá Most Popular...
Most Popular: 25,000/43,656 người dùng
Đánh giá ItemCF + Popular fallback...
ItemCF + Popular fallback: 25,000/43,656 người dùng


,model,k,group,users,Precision@K,Recall@K,NDCG@K,CatalogCoverage@K,CF_slot_fraction,popular_only_list_fraction
0,Most Popular,10,Toàn bộ,43656,0.001636,0.016218,0.007328,0.000107,0.000000,1.000000
1,Most Popular,10,Có lịch sử tích cực,2533,0.000790,0.007698,0.003110,0.000107,0.000000,1.000000
2,Most Popular,20,Toàn bộ,43656,0.001401,0.027770,0.010290,0.000214,0.000000,1.000000
3,Most Popular,20,Có lịch sử tích cực,2533,0.000770,0.015002,0.004996,0.000214,0.000000,1.000000
4,ItemCF + Popular fallback,10,Toàn bộ,43656,0.001645,0.016298,0.007405,0.058720,0.026986,0.959387
5,ItemCF + Popular fallback,10,Có lịch sử tích cực,2533,0.000947,0.009080,0.004426,0.058720,0.465101,0.300039
6,ItemCF + Popular fallback,20,Toàn bộ,43656,0.001405,0.027850,0.010362,0.063285,0.017439,0.959387
7,ItemCF + Popular fallback,20,Có lịch sử tích cực,2533,0.000849,0.016384,0.006241,0.063285,0.300553,0.300039


## 7. Chênh lệch NDCG@10 và khoảng tin cậy

Lấy chênh lệch NDCG@10 giữa hai mô hình **trên cùng người dùng**, rồi bootstrap theo người dùng 1.000 lần với seed cố định. Báo cáo khoảng percentile 95% cho trung bình chênh lệch. Nếu khoảng này chứa 0, chưa có bằng chứng rõ về cải thiện NDCG@10 trong phép đánh giá này.

Khoảng tin cậy mô tả độ bất định khi lấy mẫu người dùng trong test hiện tại, không đo bất định khi huấn luyện lại hoặc thay đổi giai đoạn thời gian. Không dùng kết quả này để chọn lại tham số.


In [9]:
bootstrap_results = paired_bootstrap_delta(
    test_cf_rows, test_popular_rows, k=PRIMARY_K,
    n_resamples=BOOTSTRAP_RESAMPLES, seed=SEED)
bootstrap_results.to_csv(REPORT_DIR / 'test_ndcg10_paired_bootstrap.csv',
                         index=False, encoding='utf-8-sig')
display(bootstrap_results)


,group,users,k,mean_NDCG_delta_CF_minus_Popular,CI95_low,CI95_high,bootstrap_resamples,CI_contains_zero
0,Toàn bộ,43656,10,0.000076,-0.000014,0.000178,1000,True
1,Có lịch sử tích cực,2533,10,0.001315,-0.000311,0.003128,1000,True


## 8. Lưu mô hình, kiểm tra nạp lại và tạo ví dụ Top-K

Thư mục mô hình chứa ma trận tương tác, ma trận lân cận, ánh xạ ID, độ phổ biến và lịch sử đã đánh giá để loại đúng sản phẩm khi dự đoán. Sau khi nạp lại, kiểm tra danh sách khuyến nghị của một số người dùng phải giống trước khi lưu.

Ví dụ chọn một người có ứng viên CF nếu có. Tên nguồn `ItemCF` hoặc `Most Popular fallback` giúp nhận biết danh sách được hình thành thế nào; không phải lời giải thích bằng LLM. Người mới nhận Most Popular theo quy tắc đã định.


In [10]:
final_model.save(MODEL_DIR, extra_config={
    'selection_sha256': selection_sha256,
    'selected_using': 'validation', 'fitted_on': 'train + validation',
    'test_used_for_training': False,
    'split_config': split_config, 'versions': selection['versions']})

restored_model = ItemCFModel.load(MODEL_DIR)
checks = list(final_model.user_ids[:5]) + ['__new_user__']
for user in checks:
    assert restored_model.recommend(user, 20) == final_model.recommend(user, 20)
    assert restored_model.recommend_popular(user, 20) == final_model.recommend_popular(user, 20)
    assert restored_model.recommend(user, 10) == restored_model.recommend(user, 20)[:10]
assert restored_model.recommend('__new_user__', 20) == final_model.recommend_popular('__new_user__', 20)
assert (restored_model.S != final_model.S).nnz == 0
del restored_model
gc.collect()

cf_demo_candidates = test_cf_rows[
    (test_cf_rows.k == max(K_VALUES)) & (test_cf_rows.cf_count > 0)]
demo_user = cf_demo_candidates.iloc[0]['user_id'] if not cf_demo_candidates.empty else next(iter(test_truth))
demo_rows = []
for user in [demo_user, '__new_user__']:
    for k in K_VALUES:
        recs, count = final_model.recommend(user, k, return_info=True)
        demo_rows.extend({'user_id': user, 'k': k, 'rank': rank,
                          'parent_asin': item,
                          'source': 'ItemCF' if rank <= count else 'Most Popular fallback'}
                         for rank, item in enumerate(recs, start=1))
demo = pd.DataFrame(demo_rows)
demo.to_csv(REPORT_DIR / 'demo_top_k.csv', index=False, encoding='utf-8-sig')
display(demo[(demo.user_id == demo_user) & (demo.k == PRIMARY_K)])
print('Đã kiểm tra lưu/nạp lại mô hình và lưu demo_top_k.csv.')


,user_id,k,rank,parent_asin,source
0,AHOUEEGPXZY52O7RHMG5VR76IKJA,10,1,B00ZORYBVI,ItemCF
1,AHOUEEGPXZY52O7RHMG5VR76IKJA,10,2,B085BB7B1M,Most Popular fallback
2,AHOUEEGPXZY52O7RHMG5VR76IKJA,10,3,B0BM4GX6TT,Most Popular fallback
3,AHOUEEGPXZY52O7RHMG5VR76IKJA,10,4,B07C533XCW,Most Popular fallback
4,AHOUEEGPXZY52O7RHMG5VR76IKJA,10,5,B09X9BG4FC,Most Popular fallback
5,AHOUEEGPXZY52O7RHMG5VR76IKJA,10,6,B00R1TAN7I,Most Popular fallback
6,AHOUEEGPXZY52O7RHMG5VR76IKJA,10,7,B0C9CWKY9G,Most Popular fallback
7,AHOUEEGPXZY52O7RHMG5VR76IKJA,10,8,B01M1OFZOG,Most Popular fallback
8,AHOUEEGPXZY52O7RHMG5VR76IKJA,10,9,B0107QYW14,Most Popular fallback
9,AHOUEEGPXZY52O7RHMG5VR76IKJA,10,10,B09C2ZW2GL,Most Popular fallback


Đã kiểm tra lưu/nạp lại mô hình và lưu demo_top_k.csv.


## 9. Tạo bản tổng hợp kết quả

`final_summary.md` tự điền các số vừa tính và diễn giải chênh lệch NDCG@10. Dùng cùng các tệp CSV để viết phần thực nghiệm trong báo cáo. Chỉ số offline từ review cho biết khả năng khôi phục sản phẩm được đánh giá tích cực; muốn kết luận về doanh thu, lượt mua hoặc hiệu quả cho cửa hàng cần dữ liệu và thí nghiệm kinh doanh tương ứng.


In [11]:
def markdown_table(frame):
    columns = [str(c) for c in frame.columns]
    lines = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
    for row in frame.itertuples(index=False, name=None):
        lines.append('| ' + ' | '.join(str(v).replace('|', '/') for v in row) + ' |')
    return '\n'.join(lines)


display_table = test_results[['model', 'k', 'group', 'users', 'Precision@K',
                              'Recall@K', 'NDCG@K', 'CatalogCoverage@K']].copy()
for column in ['Precision@K', 'Recall@K', 'NDCG@K']:
    display_table[column] = display_table[column].map(lambda x: f'{x:.8f}')
display_table['CatalogCoverage@K'] = display_table['CatalogCoverage@K'].map(lambda x: f'{100*x:.4f}%')

all_ci = bootstrap_results[bootstrap_results['group'] == 'Toàn bộ'].iloc[0]
if all_ci['CI95_low'] <= 0 <= all_ci['CI95_high']:
    ndcg_conclusion = 'Khoảng tin cậy chứa 0: chưa có bằng chứng rõ về cải thiện NDCG@10 trong phép đánh giá này.'
elif all_ci['CI95_low'] > 0:
    ndcg_conclusion = 'Khoảng tin cậy nằm trên 0: ItemCF có NDCG@10 cao hơn Most Popular trong phép đánh giá này.'
else:
    ndcg_conclusion = 'Khoảng tin cậy nằm dưới 0: ItemCF có NDCG@10 thấp hơn Most Popular trong phép đánh giá này.'

all_cf = test_results[(test_results.model == CF_LABEL) & (test_results.k == PRIMARY_K)
                      & (test_results['group'] == 'Toàn bộ')].iloc[0]
all_pop = test_results[(test_results.model == POPULAR_LABEL) & (test_results.k == PRIMARY_K)
                       & (test_results['group'] == 'Toàn bộ')].iloc[0]
unknown_fraction = (test_diagnostics['positive_pairs_with_items_unseen_in_history']
                    / test_diagnostics['novel_positive_pairs'])
no_history_fraction = 1 - test_diagnostics['users_with_positive_history'] / test_diagnostics['users']

report = f'''# Kết quả hướng 1 — Lọc cộng tác\n\nNgười thực hiện: Cường Quốc.\n\n'''
report += f'''## Thiết kế thí nghiệm\n\n- Chia theo thời gian toàn cục; giữ nguyên mốc đã lưu ở notebook 01.\n- Tương tác tích cực: rating >= {POSITIVE_THRESHOLD:g}; mỗi cặp user–item tích cực đóng góp một lần vào mô hình.\n- Thử N = {list(NEIGHBOR_GRID)} trên validation; chọn theo NDCG@10 toàn bộ người dùng, rồi Recall@10, rồi N nhỏ hơn.\n- Cấu hình chọn: N = {final_model.n_neighbors}. Chốt trước khi đọc test và huấn luyện lại trên train + validation.\n- Lịch sử cuối gồm {final_training_info['rows']:,} review và {final_training_info['catalog_items']:,} sản phẩm.\n- Loại sản phẩm từng được người dùng đánh giá trong lịch sử, ở bất kỳ mức sao nào.\n- Giữ sản phẩm chưa xuất hiện trong lịch sử ở ground truth; mô hình không gợi ý được thì tính miss.\n- Precision@K chia cho K; Recall@K chia cho số sản phẩm liên quan; NDCG@K dùng relevance nhị phân. Trung bình theo người dùng.\n- Nhóm toàn bộ gồm người có ít nhất một sản phẩm tích cực mới trong test; nhóm có lịch sử tích cực là tập con đã có ít nhất một cặp tích cực trong train + validation.\n- Lịch sử và mô hình được cố định trong suốt giai đoạn test.\n\n'''
report += '## Kết quả test\n\n' + markdown_table(display_table) + '\n\n'
report += f'''## Chênh lệch và khả năng áp dụng\n\n- Chênh lệch NDCG@10 (CF trừ Most Popular), toàn bộ người dùng: {all_ci['mean_NDCG_delta_CF_minus_Popular']:.8f}.\n- Bootstrap theo người dùng {BOOTSTRAP_RESAMPLES:,} lần, seed {SEED}; khoảng percentile 95%: [{all_ci['CI95_low']:.8f}, {all_ci['CI95_high']:.8f}]. {ndcg_conclusion}\n- Độ phủ danh mục ở K=10: Most Popular {100*all_pop['CatalogCoverage@K']:.4f}%, CF {100*all_cf['CatalogCoverage@K']:.4f}%. Đây là tỷ lệ sản phẩm xuất hiện trong ít nhất một danh sách trên danh mục lịch sử của giai đoạn test.\n- {100*no_history_fraction:.2f}% người dùng được đánh giá không có lịch sử tích cực; {100*all_cf['popular_only_list_fraction']:.2f}% danh sách CF ở K=10 hoàn toàn dùng Most Popular.\n- {100*unknown_fraction:.2f}% cặp tích cực mới trong test có sản phẩm chưa xuất hiện trong lịch sử; CF hiện tại không có cách đưa các sản phẩm đó vào danh sách.\n- Khoảng tin cậy chỉ mô tả lấy mẫu người dùng test hiện tại, không bao gồm bất định khi huấn luyện lại hoặc thay đổi thời gian.\n- Độ phủ cao hơn là dấu hiệu danh sách đa dạng hơn theo sản phẩm; chưa chứng minh công bằng theo cửa hàng hoặc cải thiện doanh thu. Dữ liệu đang dùng là review và thời điểm review.\n\n'''
report += '''## Bàn giao và bước tiếp theo\n\n- `selected_config.json`: tham số và quy tắc chọn trên validation.\n- `validation_grid.csv`: toàn bộ cấu hình đã thử.\n- `test_comparison.csv`: bảng kết quả cuối của hai mô hình, Top-10/Top-20 và hai nhóm người dùng.\n- `test_per_user.csv.gz`: chỉ số từng người dùng để đối chiếu.\n- `test_ndcg10_paired_bootstrap.csv`: chênh lệch NDCG@10 và khoảng tin cậy.\n- `test_diagnostics.json`: quy mô, mốc thời gian, sản phẩm chưa thấy và mã kiểm tra dữ liệu.\n- `demo_top_k.csv`: ví dụ khuyến nghị có ghi nguồn CF hoặc Most Popular.\n- `models/item_cf_final/`: ma trận, ánh xạ, độ phổ biến, lịch sử đã đánh giá và cấu hình. Trong notebook, chạy ô khai báo lớp trước khi gọi `ItemCFModel.load(MODEL_DIR)`.\n\nPhần CF tạo chuẩn so sánh cho các hướng đồ thị tri thức và LLM. Các hướng sau cần giữ cùng định nghĩa relevance, mốc chia thời gian, nhóm đánh giá và chính sách tập ứng viên, hoặc báo cáo rõ phần thay đổi. Sản phẩm mới và người dùng ít lịch sử là những trường hợp cần nghiên cứu thêm bằng thông tin sản phẩm.\n'''

(REPORT_DIR / 'final_summary.md').write_text(report, encoding='utf-8')
print(ndcg_conclusion)
print('Đã lưu bảng so sánh, chẩn đoán, ví dụ và báo cáo tại:', REPORT_DIR)
print('Đã lưu mô hình tại:', MODEL_DIR)


Khoảng tin cậy chứa 0: chưa có bằng chứng rõ về cải thiện NDCG@10 trong phép đánh giá này.
Đã lưu bảng so sánh, chẩn đoán, ví dụ và báo cáo tại: d:\Tài Liệu\RecommenderSystem\reports\final_cf
Đã lưu mô hình tại: d:\Tài Liệu\RecommenderSystem\models\item_cf_final
